# Speech Emotion Recognition — Speech Transformers & Self-Supervised Learning (SSL) Models

**Notebook 4 of 4** in the RAVDESS series.

| Notebook | Focus |
|----------|-------|
| 1 | Classical ML on hand-crafted features (MFCCs, Chroma, Spectral) |
| 2 | CNN on mel-spectrograms — regularization, augmentation, transfer learning |
| 3 | CNN + sequential and attention modules — BiLSTM, Attention Pooling, MHSA |
| **4** | **Speech Transformers & Self-Supervised Learning (SSL) Models — Wav2Vec 2.0, HuBERT, WavLM, Whisper** |

### Research Question

> How do modern Self-Supervised Learning (SSL) speech representation models compare to standard 2D image CNNs for Speech Emotion Recognition, both in zero-shot feature extraction and fine-tuned settings?

In Notebook 2, we saw that transfer learning from an ImageNet-pretrained 2D CNN (ResNet18) reached **87.04%** accuracy. However, visual spectrogram representations lack speech-specific priors. This notebook evaluates models pre-trained on raw speech waveforms (SSL models) to test if learning from raw audio waveforms improves classification.

### Models Evaluated

| # | Model | Pre-training Paradigm | Key Advantage |
|---|-------|-----------------------|---------------|
| 1 | **Wav2Vec 2.0** | Contrastive predictive coding | Pioneer of raw audio SSL representations |
| 2 | **HuBERT** | Masked Language Modeling over K-Means units | Learns clean phonetic and acoustic codebooks |
| 3 | **WavLM** | Masked MLM + speech denoising / speaker overlap simulation | State-of-the-Art for prosody & speaker traits |
| 4 | **Whisper** | Large-scale supervised multitasking | Supervised baseline trained on 680k hours of audio |

### Evaluation Framework

For each model, we will perform two tests:
1. **Frozen Encoder (Zero-Shot Linear Probing)**: We freeze the pre-trained backbone, extract frame-level sequence embeddings, and train a shallow classification head on top. This tests the quality of the frozen representation.
2. **Full Fine-Tuning**: We unfreeze the backbone weights and train the entire network end-to-end to adapt the representations directly to emotional classification.

## 0. Environment Setup

Installs Hugging Face's `transformers`, `datasets`, and audio-processing libraries like `soundfile` and `librosa`.

In [ ]:
!pip install transformers datasets accelerate soundfile librosa -q

In [ ]:
import os
import random
import warnings
import json

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Loading & Resampling (16kHz)

> **Crucial Acoustic Note**: Speech Transformers (Wav2Vec 2.0, HuBERT, WavLM) are pre-trained on speech sampled at **16,000 Hz**. In previous notebooks, audio was processed at 22,050 Hz. We resample all audio to 16,000 Hz to prevent domain shift and severe model degradation.

In [ ]:
# Configuration
TARGET_SR = 16000
DURATION = 4  # seconds
MAX_SAMPLES = TARGET_SR * DURATION
data_source_path = '/kaggle/input'
if not os.path.exists(data_source_path):
    data_source_path = './data'

# Scan Directory for WAV files
wav_files = []
for dirname, _, filenames in os.walk(data_source_path):
    for filename in filenames:
        if filename.endswith('.wav'):
            wav_files.append(os.path.join(dirname, filename))

print(f"Total audio files found: {len(wav_files):,}")

# Map RAVDESS Codes to Emotion Labels
EMOTIONS_MAP = {
    "01": "neutral",  "02": "calm",     "03": "happy",
    "04": "sad",      "05": "angry",    "06": "fearful",
    "07": "disgust",  "08": "surprised"
}

def get_emotion(path: str) -> str:
    code = os.path.basename(path).split("-")[2]
    return EMOTIONS_MAP.get(code, "unknown")

columns = {
    "path": wav_files,
    "emotion": [get_emotion(path) for path in wav_files]
}

df = pd.DataFrame(columns)
df = df[df["emotion"] != "unknown"].reset_index(drop=True)
print(f"Dataset shape: {df.shape}")

# Load & Resample Waveforms to 16,000 Hz in RAM
X_raw = []
for path in tqdm(df["path"], desc="Loading raw waveforms at 16kHz"):
    signal, sr = librosa.load(path, sr=TARGET_SR)
    if len(signal) < MAX_SAMPLES:
        signal = np.pad(signal, (0, MAX_SAMPLES - len(signal)))
    else:
        signal = signal[:MAX_SAMPLES]
    X_raw.append(signal)

X_raw = np.array(X_raw, dtype=np.float32)

# Encode Class Labels
encoder = LabelEncoder()
y = encoder.fit_transform(df["emotion"])
classes = list(encoder.classes_)
print(f"Loaded {len(X_raw)} waveforms of shape {X_raw.shape[1]:,}. Target classes: {classes}")

# Create Stratified Train / Val / Test Splits (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.30, stratify=y, random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print(f"Train samples: {X_train.shape[0]} | Val samples: {X_val.shape[0]} | Test samples: {X_test.shape[0]}")

## 2. Shared Evaluation & Fine-Tuning Pipeline

Common utilities to handle frozen feature extraction, training loops, early stopping, and zero-shot probing across all 4 architectures.

In [ ]:
results = {}  # Store results: results[model_name] = {'train_acc': ..., 'test_acc': ..., 'gap': ...}

# Custom Dataset Wrapper
class AudioDataset(Dataset):
    """Plain dataset — shape (raw_waveform,)."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        
    def __len__(self):  
        return len(self.X)
        
    def __getitem__(self, idx): 
        return self.X[idx], self.y[idx]

# Dynamic Hugging Face Collator Generator
def make_collate_fn(processor, model_type="ssl"):
    """
    Creates a collate function for PyTorch DataLoader using Hugging Face processors.
    model_type: 'ssl' (Wav2Vec2/HuBERT/WavLM) or 'whisper'
    """
    def collate_fn(batch):
        waveforms = [item[0].numpy() for item in batch]
        labels = [item[1].item() for item in batch]
        
        if model_type == "whisper":
            # Whisper processor converts raw audio into Mel-Spectrogram features
            inputs = processor(waveforms, sampling_rate=16000, return_tensors="pt")
            return {
                "input_features": inputs.input_features,
                "labels": torch.tensor(labels)
            }
        else:
            # SSL models expect normalized raw waveform sequences and attention masks
            inputs = processor(waveforms, sampling_rate=16000, padding=True, return_tensors="pt")
            batch_dict = {
                "input_values": inputs.input_values,
                "labels": torch.tensor(labels)
            }
            if "attention_mask" in inputs:
                batch_dict["attention_mask"] = inputs.attention_mask
            return batch_dict
            
    return collate_fn

def count_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Parameters: {trainable_params:,} | Total Parameters: {total_params:,} | Trainable %: {100 * trainable_params / total_params:.4f}%")

# Model Training Loop with Early Stopping
def train_model(model, train_loader, val_loader, optimizer, scheduler=None, 
                criterion=None, num_epochs=30, patience=5, model_name="model", model_type="ssl"):
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
        
    best_val_loss = float("inf")
    counter = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    ckpt_path = f"{model_name}_best.pth"
    
    for epoch in range(num_epochs):
        # ---- Train ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            labels = batch["labels"].to(device)
            
            if model_type == "whisper":
                input_features = batch["input_features"].to(device)
                outputs = model(input_features=input_features)
            else:
                input_values = batch["input_values"].to(device)
                attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None
                outputs = model(input_values=input_values, attention_mask=attention_mask)
                
            logits = outputs.logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        
        # ---- Validate ----
        model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].to(device)
                
                if model_type == "whisper":
                    input_features = batch["input_features"].to(device)
                    outputs = model(input_features=input_features)
                else:
                    input_values = batch["input_values"].to(device)
                    attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None
                    outputs = model(input_values=input_values, attention_mask=attention_mask)
                    
                logits = outputs.logits
                loss = criterion(logits, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(logits, 1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
                
        val_loss = running_loss / len(val_loader)
        val_acc = correct / total
        
        if scheduler is not None:
            scheduler.step(val_loss)
            
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
              
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1} (best val loss={best_val_loss:.4f})")
                break
                
    model.load_state_dict(torch.load(ckpt_path))
    return model, history

# Standardized Model Evaluation
@torch.no_grad()
def evaluate_model(model, loader, model_type="ssl"):
    model.eval()
    y_true, y_pred = [], []
    
    for batch in loader:
        labels = batch["labels"]
        
        if model_type == "whisper":
            input_features = batch["input_features"].to(device)
            outputs = model(input_features=input_features)
        else:
            input_values = batch["input_values"].to(device)
            attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None
            outputs = model(input_values=input_values, attention_mask=attention_mask)
            
        logits = outputs.logits
        _, predicted = torch.max(logits, 1)
        
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())
        
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    return acc, y_true, y_pred

## 3. Model 1 — Wav2Vec 2.0

* **Paper**: ["wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations" (Baevski et al., NeurIPS 2020)](https://arxiv.org/abs/2006.11477)
* **Core Concept**: Uses a CNN encoder to project raw audio to latent features, which are then masked. A Transformer learns context representations over masked segments by solving a contrastive task against quantized representations of the raw features.

### Key Concepts to Understand About Wav2Vec 2.0:

1. **Self-Supervised Learning (SSL) Paradigm**
   Learns speech representations from thousands of hours of unlabeled raw audio waveforms without human transcripts.
2. **1D Temporal CNN Encoder (Local Context)**
   Downsamples 16kHz audio waves to feature representations at 50Hz (20ms slices).
3. **Transformer Context Network (Global Context)**
   BERT-style multi-head self-attention network capturing long-range phonological and prosodic dynamics.
4. **Vector Quantization (The Codebook)**
   Gumbel-Softmax sampler maps continuous representations to discrete codebook entries.
5. **Contrastive Masked Pre-training**
   Masks ~49% of time steps and predicts the true quantized sound among 100 distractors.
6. **Downstream Adaptation (Linear Probing / Fine-Tuning)**
   Attaches a sequence classification head on top of the contextual representations.

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

model_id_w2v2 = "facebook/wav2vec2-base"
processor_w2v2 = Wav2Vec2Processor.from_pretrained(model_id_w2v2)

# Initialize Datasets and Dataloaders
collate_fn_w2v2 = make_collate_fn(processor_w2v2, model_type="ssl")
train_dataset = AudioDataset(X_train, y_train)
val_dataset = AudioDataset(X_val, y_val)
test_dataset = AudioDataset(X_test, y_test)

train_loader_w2v2 = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_w2v2)
val_loader_w2v2 = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_w2v2)
test_loader_w2v2 = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_w2v2)

# Test 1: Frozen Backbone (Linear Probing)
print("--- Training Wav2Vec 2.0 in FROZEN (Probing) Mode ---")
model_w2v2_frozen = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_id_w2v2, 
    num_labels=len(classes)
).to(device)

for param in model_w2v2_frozen.wav2vec2.parameters():
    param.requires_grad = False

count_parameters(model_w2v2_frozen)
optimizer = torch.optim.AdamW([p for p in model_w2v2_frozen.parameters() if p.requires_grad], lr=1e-3)

model_w2v2_frozen, history_w2v2_frozen = train_model(
    model_w2v2_frozen, train_loader_w2v2, val_loader_w2v2, optimizer,
    num_epochs=45, patience=8, model_name="wav2vec2_frozen", model_type="ssl"
)

acc_w2v2_frozen, yt_w2v2_frozen, yp_w2v2_frozen = evaluate_model(model_w2v2_frozen, test_loader_w2v2, model_type="ssl")
results["Wav2Vec 2.0 (Frozen)"] = {
    "train_acc": history_w2v2_frozen["train_acc"][-1],
    "test_acc": acc_w2v2_frozen,
    "gap": history_w2v2_frozen["train_acc"][-1] - acc_w2v2_frozen
}
print(f"Wav2Vec 2.0 (Frozen) Test Accuracy: {acc_w2v2_frozen:.4f}")

# Test 2: Full Fine-Tuning
print("\n--- Training Wav2Vec 2.0 in FINE-TUNED Mode ---")
model_w2v2_tuned = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_id_w2v2, 
    num_labels=len(classes)
).to(device)

optimizer = torch.optim.AdamW(model_w2v2_tuned.parameters(), lr=2e-5)
model_w2v2_tuned, history_w2v2_tuned = train_model(
    model_w2v2_tuned, train_loader_w2v2, val_loader_w2v2, optimizer,
    num_epochs=30, patience=6, model_name="wav2vec2_tuned", model_type="ssl"
)

acc_w2v2_tuned, yt_w2v2_tuned, yp_w2v2_tuned = evaluate_model(model_w2v2_tuned, test_loader_w2v2, model_type="ssl")
results["Wav2Vec 2.0 (Fine-Tuned)"] = {
    "train_acc": history_w2v2_tuned["train_acc"][-1],
    "test_acc": acc_w2v2_tuned,
    "gap": history_w2v2_tuned["train_acc"][-1] - acc_w2v2_tuned
}
print(f"Wav2Vec 2.0 (Fine-Tuned) Test Accuracy: {acc_w2v2_tuned:.4f}")

## 4. Model 2 — HuBERT

* **Paper**: ["HuBERT: Self-Supervised Speech Representation Learning by Masked Prediction of Hidden Units" (Hsu et al., IEEE/ACM TASLP 2021)](https://arxiv.org/abs/2106.07447)
* **Core Concept**: Introduces an offline k-means clustering step on acoustic features (like MFCCs or intermediate CNN states) to generate discrete target units. A BERT-like model is then trained via Cross-Entropy Loss to predict these cluster units on masked speech inputs.

In [ ]:
from transformers import AutoFeatureExtractor, HubertForSequenceClassification

model_id_hubert = "facebook/hubert-base-ls960"
processor_hubert = AutoFeatureExtractor.from_pretrained(model_id_hubert)

# 1. Dataloaders
collate_fn_hubert = make_collate_fn(processor_hubert, model_type="ssl")
train_loader_hubert = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_hubert)
val_loader_hubert = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_hubert)
test_loader_hubert = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_hubert)

# Test 1: Frozen Backbone (Linear Probing)
print("--- Training HuBERT in FROZEN (Probing) Mode ---")
model_hubert_frozen = HubertForSequenceClassification.from_pretrained(
    model_id_hubert, 
    num_labels=len(classes)
).to(device)

# Freeze the pre-trained transformer layers
for param in model_hubert_frozen.hubert.parameters():
    param.requires_grad = False

count_parameters(model_hubert_frozen)
optimizer = torch.optim.AdamW([p for p in model_hubert_frozen.parameters() if p.requires_grad], lr=1e-3)

model_hubert_frozen, history_hubert_frozen = train_model(
    model_hubert_frozen, train_loader_hubert, val_loader_hubert, optimizer,
    num_epochs=20, patience=4, model_name="hubert_frozen", model_type="ssl"
)

acc_hubert_frozen, yt_hubert_frozen, yp_hubert_frozen = evaluate_model(model_hubert_frozen, test_loader_hubert, model_type="ssl")
results["HuBERT (Frozen)"] = {
    "train_acc": history_hubert_frozen["train_acc"][-1],
    "test_acc": acc_hubert_frozen,
    "gap": history_hubert_frozen["train_acc"][-1] - acc_hubert_frozen
}
print(f"HuBERT (Frozen) Test Accuracy: {acc_hubert_frozen:.4f}")

# Test 2: Full Fine-Tuning
print("\n--- Training HuBERT in FINE-TUNED Mode ---")
model_hubert_tuned = HubertForSequenceClassification.from_pretrained(
    model_id_hubert, 
    num_labels=len(classes)
).to(device)

optimizer = torch.optim.AdamW(model_hubert_tuned.parameters(), lr=2e-5)

model_hubert_tuned, history_hubert_tuned = train_model(
    model_hubert_tuned, train_loader_hubert, val_loader_hubert, optimizer,
    num_epochs=25, patience=5, model_name="hubert_tuned", model_type="ssl"
)

acc_hubert_tuned, yt_hubert_tuned, yp_hubert_tuned = evaluate_model(model_hubert_tuned, test_loader_hubert, model_type="ssl")
results["HuBERT (Fine-Tuned)"] = {
    "train_acc": history_hubert_tuned["train_acc"][-1],
    "test_acc": acc_hubert_tuned,
    "gap": history_hubert_tuned["train_acc"][-1] - acc_hubert_tuned
}
print(f"HuBERT (Fine-Tuned) Test Accuracy: {acc_hubert_tuned:.4f}")

## 5. Model 3 — WavLM

* **Paper**: ["WavLM: Large-Scale Self-Supervised Pre-Training for Full Stack Speech Processing" (Chen et al., IEEE JSTSP 2022)](https://arxiv.org/abs/2110.11550)
* **Core Concept**: Extends HuBERT by adding a gated relative position bias and an overlapping speech/noise simulation task during pre-training. This enables the model to perform exceptionally well on non-ASR tasks like speaker identity and speaker emotion recognition.

In [ ]:
from transformers import AutoFeatureExtractor, WavLMForSequenceClassification

model_id_wavlm = "microsoft/wavlm-base-plus"
processor_wavlm = AutoFeatureExtractor.from_pretrained(model_id_wavlm)

# 1. Dataloaders
collate_fn_wavlm = make_collate_fn(processor_wavlm, model_type="ssl")
train_loader_wavlm = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_wavlm)
val_loader_wavlm = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_wavlm)
test_loader_wavlm = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_wavlm)

# Test 1: Frozen Backbone (Linear Probing)
print("--- Training WavLM in FROZEN (Probing) Mode ---")
model_wavlm_frozen = WavLMForSequenceClassification.from_pretrained(
    model_id_wavlm, 
    num_labels=len(classes)
).to(device)

# Freeze the pre-trained transformer layers
for param in model_wavlm_frozen.wavlm.parameters():
    param.requires_grad = False

count_parameters(model_wavlm_frozen)
optimizer = torch.optim.AdamW([p for p in model_wavlm_frozen.parameters() if p.requires_grad], lr=1e-3)

model_wavlm_frozen, history_wavlm_frozen = train_model(
    model_wavlm_frozen, train_loader_wavlm, val_loader_wavlm, optimizer,
    num_epochs=20, patience=4, model_name="wavlm_frozen", model_type="ssl"
)

acc_wavlm_frozen, yt_wavlm_frozen, yp_wavlm_frozen = evaluate_model(model_wavlm_frozen, test_loader_wavlm, model_type="ssl")
results["WavLM (Frozen)"] = {
    "train_acc": history_wavlm_frozen["train_acc"][-1],
    "test_acc": acc_wavlm_frozen,
    "gap": history_wavlm_frozen["train_acc"][-1] - acc_wavlm_frozen
}
print(f"WavLM (Frozen) Test Accuracy: {acc_wavlm_frozen:.4f}")

# Test 2: Full Fine-Tuning
print("\n--- Training WavLM in FINE-TUNED Mode ---")
model_wavlm_tuned = WavLMForSequenceClassification.from_pretrained(
    model_id_wavlm, 
    num_labels=len(classes)
).to(device)

optimizer = torch.optim.AdamW(model_wavlm_tuned.parameters(), lr=2e-5)

model_wavlm_tuned, history_wavlm_tuned = train_model(
    model_wavlm_tuned, train_loader_wavlm, val_loader_wavlm, optimizer,
    num_epochs=25, patience=4, model_name="wavlm_tuned", model_type="ssl"
)

acc_wavlm_tuned, yt_wavlm_tuned, yp_wavlm_tuned = evaluate_model(model_wavlm_tuned, test_loader_wavlm, model_type="ssl")
results["WavLM (Fine-Tuned)"] = {
    "train_acc": history_wavlm_tuned["train_acc"][-1],
    "test_acc": acc_wavlm_tuned,
    "gap": history_wavlm_tuned["train_acc"][-1] - acc_wavlm_tuned
}
print(f"WavLM (Fine-Tuned) Test Accuracy: {acc_wavlm_tuned:.4f}")

## 6. Model 4 — Whisper (Encoder-Only Baseline)

* **Paper**: ["Robust Speech Recognition via Large-Scale Weak Supervision" (Radford et al., 2022)](https://arxiv.org/abs/2212.04356)
* **Core Concept**: Whisper is trained using supervised multitasking (ASR, translation, voice activity detection) on 680,000 hours of weakly labeled audio. We evaluate if supervised representation learning offers different features than SSL representation learning.

In [ ]:
from transformers import AutoFeatureExtractor, WhisperForAudioClassification

model_id_whisper = "openai/whisper-base"
processor_whisper = AutoFeatureExtractor.from_pretrained(model_id_whisper)

# 1. Dataloaders with Whisper mel-spectrogram collate function
collate_fn_whisper = make_collate_fn(processor_whisper, model_type="whisper")
train_loader_whisper = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn_whisper)
val_loader_whisper = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_whisper)
test_loader_whisper = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn_whisper)

# Test 1: Frozen Backbone (Linear Probing)
print("--- Training Whisper in FROZEN (Probing) Mode ---")
model_whisper_frozen = WhisperForAudioClassification.from_pretrained(
    model_id_whisper, 
    num_labels=len(classes)
).to(device)

# Freeze the pre-trained encoder layers
for param in model_whisper_frozen.whisper.parameters():
    param.requires_grad = False

count_parameters(model_whisper_frozen)
optimizer = torch.optim.AdamW([p for p in model_whisper_frozen.parameters() if p.requires_grad], lr=1e-3)

model_whisper_frozen, history_whisper_frozen = train_model(
    model_whisper_frozen, train_loader_whisper, val_loader_whisper, optimizer,
    num_epochs=20, patience=4, model_name="whisper_frozen", model_type="whisper"
)

acc_whisper_frozen, yt_whisper_frozen, yp_whisper_frozen = evaluate_model(model_whisper_frozen, test_loader_whisper, model_type="whisper")
results["Whisper (Frozen)"] = {
    "train_acc": history_whisper_frozen["train_acc"][-1],
    "test_acc": acc_whisper_frozen,
    "gap": history_whisper_frozen["train_acc"][-1] - acc_whisper_frozen
}
print(f"Whisper (Frozen) Test Accuracy: {acc_whisper_frozen:.4f}")

# Test 2: Full Fine-Tuning
print("\n--- Training Whisper in FINE-TUNED Mode ---")
model_whisper_tuned = WhisperForAudioClassification.from_pretrained(
    model_id_whisper, 
    num_labels=len(classes)
).to(device)

optimizer = torch.optim.AdamW(model_whisper_tuned.parameters(), lr=1e-5)

model_whisper_tuned, history_whisper_tuned = train_model(
    model_whisper_tuned, train_loader_whisper, val_loader_whisper, optimizer,
    num_epochs=25, patience=4, model_name="whisper_tuned", model_type="whisper"
)

acc_whisper_tuned, yt_whisper_tuned, yp_whisper_tuned = evaluate_model(model_whisper_tuned, test_loader_whisper, model_type="whisper")
results["Whisper (Fine-Tuned)"] = {
    "train_acc": history_whisper_tuned["train_acc"][-1],
    "test_acc": acc_whisper_tuned,
    "gap": history_whisper_tuned["train_acc"][-1] - acc_whisper_tuned
}
print(f"Whisper (Fine-Tuned) Test Accuracy: {acc_whisper_tuned:.4f}")

## 7. Leaderboard & Final Comparative Discussion

Compare the zero-shot probing and fine-tuning test accuracy of Wav2Vec 2.0, HuBERT, WavLM, and Whisper. Discussion on why certain models perform better on non-ASR semantic voice tasks.

In [ ]:
# Convert the results dictionary into a Pandas DataFrame
comparison_df = pd.DataFrame(results).T

if not comparison_df.empty:
    # Rename columns for presentation
    comparison_df = comparison_df.rename(columns={
        "train_acc": "Train Accuracy",
        "test_acc": "Test Accuracy",
        "gap": "Train-Test Gap"
    })
    
    # Print the Leaderboard Table
    print("====================================================================")
    print("                        MODEL LEADERBOARD")
    print("====================================================================")
    display(comparison_df.round(4))
    print("\n")
    
    # Create Accuracy and Overfitting Gap Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.set_theme(style="whitegrid")
    
    # Plot 1: Test Accuracy Comparison (Higher is Better)
    bars1 = sns.barplot(
        x=comparison_df.index, 
        y=comparison_df["Test Accuracy"], 
        ax=axes[0], 
        palette="muted"
    )
    axes[0].set_title("Test Accuracy Comparison (Higher is Better)", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis='x', rotation=25)
    
    # Add values on top of the bars
    for bar in bars1.patches:
        val = bar.get_height()
        axes[0].text(
            bar.get_x() + bar.get_width()/2, 
            val + 0.01, 
            f"{val:.2%}", 
            ha="center", 
            va="bottom", 
            fontsize=9
        )
        
    # Plot 2: Overfitting Gap (Lower is Better)
    bars2 = sns.barplot(
        x=comparison_df.index, 
        y=comparison_df["Train-Test Gap"], 
        ax=axes[1], 
        palette="coolwarm"
    )
    axes[1].set_title("Overfitting Gap: Train - Test (Lower is Better)", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Accuracy Gap")
    axes[1].set_ylim(min(0, comparison_df["Train-Test Gap"].min() - 0.05), max(0.4, comparison_df["Train-Test Gap"].max() + 0.05))
    axes[1].tick_params(axis='x', rotation=25)
    
    # Add values on top of the bars
    for bar in bars2.patches:
        val = bar.get_height()
        axes[1].text(
            bar.get_x() + bar.get_width()/2, 
            val + 0.005 if val >= 0 else val - 0.015, 
            f"{val:.2%}", 
            ha="center", 
            va="bottom" if val >= 0 else "top", 
            fontsize=9
        )
        
    plt.tight_layout()
    plt.show()
else:
    print("No results found in the 'results' dictionary yet. Please run and evaluate at least one model.")